<a href="https://colab.research.google.com/github/pksheaad/Transformers/blob/main/04_implementing_a_transformer_block_PK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# importing dependencies
import requests
from typing import Dict, List, Any

import torch
import torch.nn as nn
from torch.utils.data.dataset import Dataset
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import RandomSampler

In [2]:
# Set the device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [3]:
# Retrieve the data
url = "https://raw.githubusercontent.com/pksheaad/Transformers/refs/heads/main/Data/tiny-shakespeare.txt"

request = requests.get(url = url)
text = request.text
text[:50]

'First Citizen:\nBefore we proceed any further, hear'

In [14]:
# creating encoding and decoding
class CharTokenizer:
  def __init__(self, vocabulary) -> None:
    self.token_ids_for_char = {char : token_ids for token_ids, char in enumerate(vocabulary)}
    self.char_for_token_ids = {token_ids : char for token_ids, char in enumerate(vocabulary) }

  @staticmethod
  def text_to_test(text):
    vocabulary = set(text)
    return CharTokenizer(sorted(list(vocabulary)))

  def encoder(self,text):
    token_ids_list = []
    for char in text:
      token_ids_list.append(self.token_ids_for_char[char])

    return torch.tensor(token_ids_list, dtype = torch.long)

  def decoder(self, token_ids):
    char_list = []
    for token_id in token_ids.tolist():
      char_list.append(self.char_for_token_ids[token_id])

    return "".join(char_list)

  def get_length(self):
    return len(self.token_ids_for_char)

In [18]:
# testing
print(text[:50])
tokenizer = CharTokenizer.text_to_test(text = text)
#print(tokenizer.encoder(text[:50]))
#print(tokenizer.decoder(tokenizer.encoder(text[:50])))
print(tokenizer.get_length())


First Citizen:
Before we proceed any further, hear
65


In [23]:
class TokenIdsDataset(Dataset):

  # Step 1 Implement the constructor
  def __init__(self, data, block_size) -> None:
    super().__init__()
    self.data = data
    self.block_size = block_size

  # Step 2 Implement the __len__ method:
  def __len__(self):
    return len(self.data) - self.block_size

  # Step 3 Implement the __getitem__ method:
  def __getitem__(self, index):

    assert index < len(self.data) - self.block_size
    x = self.data[index : index + self.block_size]
    y = self.data[index + 1 : index + 1 + self.block_size]

    return x, y



In [25]:
input = tokenizer.encoder(text = text)
dataset = TokenIdsDataset(data = input, block_size = 64)
sampler = RandomSampler(data_source = dataset, replacement = True)
dataloader = DataLoader(dataset = dataset, batch_size = 64, sampler = sampler)
x , y = next(iter(dataloader))
x[0], y[0]


(tensor([53, 61,  1, 47, 58,  1, 52, 53, 56,  1, 41, 39, 52,  1, 50, 43, 39, 56,
         52,  1, 53, 44,  1, 46, 47, 51,  8,  0,  0, 14, 17, 26, 34, 27, 24, 21,
         27, 10,  0, 20, 39, 60, 43,  1, 63, 53, 59,  1, 47, 51, 54, 53, 56, 58,
         59, 52, 43, 42,  1, 46, 47, 51,  1, 40]),
 tensor([61,  1, 47, 58,  1, 52, 53, 56,  1, 41, 39, 52,  1, 50, 43, 39, 56, 52,
          1, 53, 44,  1, 46, 47, 51,  8,  0,  0, 14, 17, 26, 34, 27, 24, 21, 27,
         10,  0, 20, 39, 60, 43,  1, 63, 53, 59,  1, 47, 51, 54, 53, 56, 58, 59,
         52, 43, 42,  1, 46, 47, 51,  1, 40, 63]))

In [26]:
# Configuration

config = {
    "vocabulary_size" : tokenizer.get_length(),
    "context_size" : 256,
    "embedding_dim" : 768,
    "num_head" : 12,
    "num_layers" : 10,
    "dropout_rate" : 0.1,
    "use_bias" : False
}

config["head_size"] = config["embedding_dim"] // config["num_head"]

In [34]:
class AttentionHead(nn.Module):
  def __init__(self, config:Dict[str, Any]) -> None:
    super().__init__()

    # Step 1==> Create Q, K and V Weight
    self.Q_Weight = nn.Linear(in_features =  config['embedding_dim'], out_features = config['head_size'], bias = config['use_bias'])

    self.K_Weight = nn.Linear(in_features =  config['embedding_dim'], out_features = config['head_size'], bias = config['use_bias'])

    self.V_Weight = nn.Linear(in_features =  config['embedding_dim'], out_features = config['head_size'], bias = config['use_bias'])

    # Step 2 ==> Casual Mask
    casual_attention_mask = torch.tril(torch.ones(config['context_size'], config['context_size']))
    self.register_buffer("casual_attention_mask", casual_attention_mask)

    # Step 3==> Dropout layer
    self.dropout = nn.Dropout(p = config['dropout_rate'])

  # Implement forward method
  def forward(self, input):
    batch_size, token_num, embedding_dim = input.shape
    Q = self.Q_Weight(input)
    K = self.K_Weight(input)
    V = self.V_Weight(input)

    # Step 1 dot product of Q and K
    attention_score = Q @ K.transpose(1,2)
    # Step 2 Masking
    attention_score = attention_score.masked_fill(self.casual_attention_mask[:token_num, :token_num]== 0, -torch.inf)
    # Step 3 Minimize the value of attention mask
    attention_score = attention_score / K.shape[-1]**0.5
    # step 4 Softmax
    attention_score = torch.softmax(attention_score, dim = -1)
    # Step 5 dot product of result of softmax and V matrix
    attention_score = attention_score @ V
    # Step 6 Dropout layer
    attention_score = self.dropout(attention_score)

    return attention_score


In [35]:
# Testing
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(input.shape)
ah = AttentionHead(config = config)
output = ah(input)
print(output.shape)

torch.Size([8, 256, 768])
torch.Size([8, 256, 64])


In [39]:
# creating MutiAttentionHead
# setp 1 import nn.Module
class MultiAttentionHead(nn.Module):
  def __init__(self, config:Dict[str, Any]) -> None:
    super().__init__()

    attention_heads = [AttentionHead(config) for _ in range(config['num_head'])]
    # step 2 store attention_heads to Module list
    self.head_list = nn.ModuleList(attention_heads)

    # step 3 Extra linear layer
    self.linear = nn.Linear(in_features = config['head_size'] * config['num_head'], out_features = config['embedding_dim'])

    # Step 4 Dropout layer for Normalization
    self.dropout = nn.Dropout(p = config['dropout_rate'])

  # Implement forward method
  def forward(self, input):
    heads = [head(input) for head in self.head_list]
    score_change = torch.cat(heads, dim = -1)
    score_change = self.linear(score_change)
    score_change = self.dropout(score_change)
    return score_change


In [40]:
# Tesing
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(input.shape)
mha = MultiAttentionHead(config = config)
output = mha(input)
print(output.shape)

torch.Size([8, 256, 768])
torch.Size([8, 256, 768])


# Implementation of Transformer Block


## Feed Forward Block
To define the Feedforward block we will cretae a class which inherit nn.Module

**Linear Layers:** The first linear layer projects the input dimension to four times its size, followed by a GELU activation function and another linear layer that reduces the dimension back to the original size.

**Dropout Layer:** Added to prevent overfitting.

In [41]:
from torch.nn.modules.linear import Linear
class FeedForward(nn.Module):
  def __init__(self, config:Dict[str, Any]) -> None:
    super().__init__()

    self.linear_layer = nn.Sequential(
        nn.Linear(in_features = config['embedding_dim'], out_features = config['embedding_dim']*4),
        nn.GELU(),
        nn.Linear(in_features =  config['embedding_dim']*4, out_features = config['embedding_dim']),
        nn.Dropout(p = config['dropout_rate'])
    )

  # Implement forward Method
  def forward(self, input):
    x = self.linear_layer(input)
    return x

In [42]:
# Testing
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(f"Input Shape: {input.shape}")
# instantiate the class
ff = FeedForward(config = config)
output = ff(input)
print(f"Output Shape: {output.shape}")

Input Shape: torch.Size([8, 256, 768])
Output Shape: torch.Size([8, 256, 768])


## Transformer Block Implementation

Since FeedForward class is implemented Now we will implement the Transoformer block



### Transformer Block Components

1. **MultiheadAttention Block**
2. **Layer Normalization** 2 Layer Normalization one before MultiheadAttenblock and another before Feedforward Block
3. **FeedForward Block**
4. **Residual Component** two residual compoenent to help the vanishing the gradient descent (one after MHA and another after Feed Forward)


In [46]:
class Block(nn.Module):
  def __init__(self,config:Dict[str, Any] ) -> None:
    super().__init__()

    # Step 1
    self.layer_norm_1 = nn.LayerNorm(config["embedding_dim"])
    # Step 2
    self.mha = MultiAttentionHead(config = config)
    # Step 3 (Forward method will implement the residual for vanishing the gradient descent)
    # Step 4
    self.layer_norm_2 = nn.LayerNorm(config["embedding_dim"])
    # Step 5
    self.ff = FeedForward(config = config)
    # Step 6 (Forward method will implement the residual for vanishing the gradient descent)


  # Implement Forward method
  def forward(self, input):
    residual = input
    x = self.layer_norm_1(input)
    x = self.mha(x)
    x = x + residual

    residual = x
    x = self.layer_norm_2(x)
    x = self.ff(x)
    x = x + residual

    return x



In [47]:
# Testing
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(f"Input Shape {input.shape}")
# instatiate the Transformaer Block
block = Block(config = config)
output = block(input)
print(f"Output Shape: {output.shape}")


Input Shape torch.Size([8, 256, 768])
Output Shape: torch.Size([8, 256, 768])
